In [2]:
from datasets import load_dataset

ds = load_dataset("JacobLinCool/VoiceBank-DEMAND-16k")

/home/ys/diploma/denoising_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch

class TestDataset(torch.utils.data.Dataset):
    def __init__(self,ds):
        super().__init__()
        self.ds = ds

    def __getitem__(self, index):
        
        return torch.tensor(self.ds[index]['noisy']['array']).to(torch.float32),  torch.tensor(self.ds[index]['clean']['array']).to(torch.float32), 
    
    def __len__(self):
        return self.ds.__len__()
    

test_set = TestDataset(ds['test'])
test_loader = torch.utils.data.DataLoader(test_set, batch_size=1, )
a = next(iter(test_loader))[0]
a.shape

torch.Size([1, 27861])

In [10]:
import cpuinfo
from time import perf_counter
from loaders import *
from lightning_modules.lightning_module import *
from utils.cfg_loader import load_cfg
import GPUtil
from torchmetrics.audio import( ScaleInvariantSignalDistortionRatio as SISDR, SignalDistortionRatio as SDR,
                                SignalNoiseRatio as SNR, ScaleInvariantSignalNoiseRatio as SISNR,
                                PerceptualEvaluationSpeechQuality as PESQ,
                                ShortTimeObjectiveIntelligibility as STOI)



cfg_path = 'configs/denoise_model_v1_cfg.yaml'
ckpt_path = '/home/ys/diploma/ckpts/version_29/checkpoints/last.ckpt'
#ckpt_path = '/home/ys/diploma/ckpts/last.ckpt'
model_cfg = load_cfg(cfg_path)

metrics = dict( pesq_wb = PESQ(16000, 'wb'),
                pesq_nb = PESQ(16000, 'nb'),
                stoi = STOI(16000),
                snratio = SNR(),
                sdratio = SDR(),
                sisdratio = SISDR(),
                sisnratio = SISNR())
                

model = UltraSpectrogramLightningModelUnet.load_from_checkpoint(ckpt_path, **model_cfg)
#model = SpectrogramLightningModelUnet.load_from_checkpoint(ckpt_path, **model_cfg)

info = cpuinfo.get_cpu_info()
gpus = GPUtil.getGPUs()
print(" "*50)
print("-"*20 + 'DEVICE INFO' + "-"*20)
print(f"Процессор: {info['brand_raw']}")
print(f"Количество ядер: {info['count']}")
for g in gpus:
    print(f"Видеокарта: {g.name}")
    print(f"Память: {g.memoryTotal} MB")
    print(f"Используется памяти: {g.memoryUsed} MB")
    print(f"Загрузка GPU: {g.load * 100}%")
print("-"*20 + '----------' + "-"*20)   

def reset_metrics(metrics):
    """Сбрасывает все метрики перед новым вычислением."""
    for metric in metrics.values():
        metric.reset()

def compute_metrics(cleaned_wf, clean_wf, metric):
        cleaned_wf = cleaned_wf.detach().cpu()
        clean_wf = clean_wf.detach().cpu()
        cleaned_wf_shape = cleaned_wf.shape[-1]
        clean_wf_shape = clean_wf.shape[-1]
        if cleaned_wf.shape[1] != 1:
            cleaned_wf = cleaned_wf.sum(1, keepdims=True)
        if clean_wf.shape[1] != 1:
            clean_wf = clean_wf.sum(1, keepdims=True)
        if clean_wf_shape == min(clean_wf_shape, cleaned_wf_shape):
            cleaned_wf = cleaned_wf[:, :, :clean_wf_shape]
        else:
            clean_wf = clean_wf[:, :, :cleaned_wf_shape]

        
        for k in metric.keys():
            metric[k].update(cleaned_wf, clean_wf) 
        
        
def bench_model(model,metrics, loader, num_iters=100):
    model.eval()
    cpu = []
    gpu = []
    reset_metrics(metrics)
    if num_iters=="max":
        num_iters=len(test_loader)
    with torch.no_grad():
        for i, batch in enumerate(loader):
            
            model = model.to('cpu')
            mixed, clean = batch
            
            mixed = mixed.to('cpu')
            
            if mixed.ndim == 2 and mixed.shape[0] > 1:
                mixed = mixed.sum(0, keepdim=True)
            
            if mixed.ndim == 3 and mixed.shape[1] > 1:
                mixed = mixed.sum(1, keepdim=True)

            if clean.ndim == 2 and clean.shape[0] > 1:
                clean = clean.sum(0, keepdim=True)
            
            if clean.ndim == 3 and clean.shape[1] > 1:
                clean = clean.sum(1, keepdim=True)

           
            
            start = perf_counter()
            out = model.run(mixed)
            delta =perf_counter() - start
            cpu.append(delta)
            
            model = model.to('cuda')
            mixed = mixed.to('cuda')
            clean = clean.to('cuda')
            
                
            start = perf_counter()

            out = model.run(mixed)
            delta = perf_counter() - start
            gpu.append(delta)

            out = out.to('cuda')

            if out.shape[-1] > clean.shape[-1]:
                out = out[..., :clean.shape[-1]]
        
            if clean.ndim != out.ndim:
                clean = clean[None, ...]

            compute_metrics(out, clean, metrics)
            
            if (i + 1) % (num_iters) == 0:
                break

        metric_values = {k: metrics[k].compute().item() for k in metrics.keys()}



    return cpu, gpu, metric_values


model


                                                  
--------------------DEVICE INFO--------------------
Процессор: Intel(R) Core(TM) i5-10300H CPU @ 2.50GHz
Количество ядер: 8
Видеокарта: NVIDIA GeForce GTX 1650
Память: 4096.0 MB
Используется памяти: 293.0 MB
Загрузка GPU: 1.0%
--------------------------------------------------


UltraSpectrogramLightningModelUnet(
  (stft): Spectrogram()
  (model): DenoisingModelUnet(
    (encoder): SpectrumEncoder(
      (encoder_features): Sequential(
        (layer_0): AdaptiveResBlock(
          (block): MobileBlock(
            (conv): Conv2d(1, 8, kernel_size=(9, 9), stride=(1, 1), padding=(4, 4))
            (depth_wise): Conv2d(8, 8, kernel_size=(9, 9), stride=(1, 1), groups=8)
            (point_wise): Conv2d(8, 8, kernel_size=(1, 1), stride=(1, 1), padding=(4, 4))
            (bn): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (act): ELU(alpha=1.0)
            (dropout): Dropout2d(p=0.2, inplace=False)
          )
          (scale): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
          (adapt_res): Conv2d(1, 8, kernel_size=(1, 1), stride=(1, 1), bias=False)
        )
        (layer_1): AdaptiveResBlock(
          (block): MobileBlock(
            (conv): Conv2d(8, 32, kernel_size=(7, 7), stri

# VoiceBank + DEMAND

In [4]:
cpu, gpu,metric_values = bench_model(model, metrics, loader=test_loader, num_iters='max')


print('-'*20 + '+CPU+' + '-'*20)
print('Avg CPU Inference [s] : ', torch.tensor(cpu).mean().item())
print('-'*20 + '-----' + '-'*20)
print('-'*20 + '+GPU+' + '-'*20)
print('Avg GPU Inference [s] : ', torch.tensor(gpu).mean().item())
print('-'*20 + '-----' + '-'*20)
print('-'*19 + 'METRICS' + '-'*19 )
print(f"SNR [dB]: {metric_values['snratio']}"),
print(f"SDR [dB]: {metric_values['sdratio']}")
print(f"SI-SDR [dB]: {metric_values['sisdratio']}")
print(f"SI-SNR [dB]: {metric_values['sisnratio']}")
print(f"STOI: {metric_values['stoi']}")
print(f"PESQ-NB: {metric_values['pesq_nb']}")
print(f"PESQ-WB: {metric_values['pesq_wb']}")

print('-'*19 + '-------' + '-'*19 )
print(" "*50)

--------------------+CPU+--------------------
Avg CPU Inference [s] :  1.20132315158844
---------------------------------------------
--------------------+GPU+--------------------
Avg GPU Inference [s] :  0.14618463814258575
---------------------------------------------
-------------------METRICS-------------------
SNR [dB]: 3.8419125080108643
SDR [dB]: 5.418313026428223
SI-SDR [dB]: 5.217263698577881
SI-SNR [dB]: 5.217719078063965
STOI: 0.9025267958641052
PESQ-NB: 2.700246572494507
PESQ-WB: 1.877081274986267
---------------------------------------------
                                                  


In [5]:
audio = list(iter(test_loader))[-456]

In [6]:
mixed = audio[0]
cleaned = model.run(mixed)

cleaned.shape

torch.Size([1, 1, 128000])

In [7]:
from IPython.display import Audio 

display(Audio(cleaned.detach().cpu().numpy()[0], rate=16000))
display(Audio(audio[0].detach().cpu().numpy()[0], rate=16000))
display(Audio(audio[1].detach().cpu().numpy()[0], rate=16000))




# LibreSpeech + Wham

In [5]:
_, _, test_loader = get_loaders(speech_dirs=["dataset/dev-clean", "dataset/test-clean"],
                                                    noise_dir="dataset/wham_noise/wham_noise",
                                                    batch_size=1,
                                                    padding_strategy=None)

#print(next(iter(test_loader))[0].shape)
cpu, gpu,metric_values = bench_model(model, metrics, loader=test_loader, num_iters='max')


print('-'*20 + '+CPU+' + '-'*20)
print('Avg CPU Inference [s] : ', torch.tensor(cpu).mean().item())
print('-'*20 + '-----' + '-'*20)
print('-'*20 + '+GPU+' + '-'*20)
print('Avg GPU Inference [s] : ', torch.tensor(gpu).mean().item())
print('-'*20 + '-----' + '-'*20)
print('-'*19 + 'METRICS' + '-'*19 )
print(f"SNR [dB]: {metric_values['snratio']}"),
print(f"SDR [dB]: {metric_values['sdratio']}")
print(f"SI-SDR [dB]: {metric_values['sisdratio']}")
print(f"SI-SNR [dB]: {metric_values['sisnratio']}")
print(f"STOI: {metric_values['stoi']}")
print(f"PESQ-NB: {metric_values['pesq_nb']}")
print(f"PESQ-WB: {metric_values['pesq_wb']}")

print('-'*19 + '-------' + '-'*19 )
print(" "*50)

--------------------+CPU+--------------------
Avg CPU Inference [s] :  1.2340052127838135
---------------------------------------------
--------------------+GPU+--------------------
Avg GPU Inference [s] :  0.2253415435552597
---------------------------------------------
-------------------METRICS-------------------
SNR [dB]: 7.965380668640137
SDR [dB]: 7.612330436706543
SI-SDR [dB]: 7.293706893920898
SI-SNR [dB]: 7.295334339141846
STOI: 0.8895750641822815
PESQ-NB: 2.3972959518432617
PESQ-WB: 1.7521833181381226
---------------------------------------------
                                                  


In [36]:
from IPython.display import Audio 
sample_batch = next(iter(test_loader))
mixed_waveforms, speech_waveforms = sample_batch

for i, (mixed_waveform, speech_waveform) in enumerate(zip(mixed_waveforms, speech_waveforms)):
    print(f"--------------------\nsample {i} from batch")
    print(f"Input audio {i + 1}:")
    display(Audio(mixed_waveform.numpy(), rate=16000))
    print(f"Target audio {i + 1}:")
    display(Audio(speech_waveform.numpy(), rate=16000))

--------------------
sample 0 from batch
Input audio 1:


Target audio 1:


In [11]:

audio = list(iter(test_loader))[-480]

mixed = audio[0]
cleaned = model.run(mixed)



display(Audio(cleaned.detach().cpu().numpy()[0], rate=16000))
display(Audio(audio[0].detach().cpu().numpy()[0], rate=16000))
display(Audio(audio[1].detach().cpu().numpy()[0], rate=16000))

In [ ]:
torchaudio.save('examples/cleaned.wav', torch.tensor(audio[0].detach().cpu()), sample_rate=16000)
torchaudio.save('examples/clean.wav', torch.tensor(cleaned[0]), sample_rate=16000)
torchaudio.save('examples/noisy.wav', torch.tensor(audio[1].detach().cpu()), sample_rate=16000)

NameError: name 'out' is not defined

In [ ]:
from torchvision.models import vgg11_bn, VGG11_BN_Weights
from torch import nn
import torch
_, valid_loader, test_loader = get_loaders(speech_dirs=["dataset/dev-clean", "dataset/test-clean"],
                                                    noise_dir="dataset/wham_noise/wham_noise",
                                                    batch_size=1,
                                                    padding_strategy=None)
aud1 = next(iter(valid_loader))[0].to('cuda')
aud2 = next(iter(test_loader))[0].to('cuda')
aud = torch.cat((aud1, aud2), dim=-1)
print(aud1.shape)
result = model.run(aud1[0])





torch.Size([1, 2, 128000])
torch.Size([1, 1, 128000])


In [10]:
from torchviz import make_dot
from torchview import draw_graph

model_graph = draw_graph(
    model.model,
    input_size=(1, 1, 1024, 1025),  
    device="cuda",
    expand_nested=True  # Показывает внутренности UNet
)
model_graph.visual_graph  # Отображает в ноутбуке
model_graph.visual_graph.render("unet_model", format="png") 

'unet_model.png'

In [ ]:
from IPython.display import Audio 
Audio(aud[0].numpy(),rate = 16000)

In [ ]:
Audio(result,rate = 16000) 

NameError: name 'Audio' is not defined

In [ ]:
!nvidia-smi

Sun Apr 13 12:49:27 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 565.72                 Driver Version: 566.14         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650        On  |   00000000:01:00.0 Off |                  N/A |
| N/A   52C    P8              4W /   50W |     393MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from mamba_ssm import Mamba

class MambaPhaseCorrector(nn.Module):
    def __init__(self, d_model=4):
        super().__init__()
        self.conv_in = nn.Conv2d(2, d_model, kernel_size=3, padding=1)
        self.mamba = Mamba(
            d_model=d_model, 
            d_state=4,  
            d_conv=4,    
            expand=2             )
        self.conv_out = nn.Conv2d(d_model, 1, kernel_size=1)

    def forward(self, mag, phase):
        x = torch.cat([mag, phase], dim=1)  # [B, 2, F, T]
        x = self.conv_in(x)  # [B, d_model, F, T]
        
        
        B, C, F, T = x.shape
        x = x.permute(0, 2, 3, 1).reshape(B * F, T, C)  # [B*F, T, d_model]
        x = self.mamba(x)  
        x = x.reshape(B, F, T, C).permute(0, 3, 1, 2)  # [B, C, F, T]
        
        return phase + torch.tanh(self.conv_out(x))  


class TFMambaBotteleneck(nn.Module):
    def __init__(self, d_model=4):
        super().__init__()
        self.conv_in = nn.Conv2d(2, d_model, kernel_size=3, padding=1)
        self.mamba = Mamba(
            d_model=d_model, 
            d_state=4,  
            d_conv=4,    
            expand=2             )
        self.conv_out = nn.Conv2d(d_model, 1, kernel_size=1)

    def forward(self, mag, encoded_audio):
        x = torch.cat([mag, encoded_audio], dim=1)  # [B, 2, F, T]
        x = self.conv_in(x)  # [B, d_model, F, T]
        
        
        B, C, F, T = x.shape
        x = x.permute(0, 2, 3, 1).reshape(B * F, T, C)  # [B*F, T, d_model]
        x = self.mamba(x)  
        x = x.reshape(B, F, T, C).permute(0, 3, 1, 2)  # [B, C, F, T]
        
        return encoded_audio + torch.tanh(self.conv_out(x))  
    
class EncoderAudio(nn.Module):
    def __init__(self, kernel_size = 3, hidden_dim = 256, out_dim=512, ):
        super().__init__()
        self.input_l = nn.Conv1d(1, hidden_dim, kernel_size=1)
        self.mamba = Mamba(d_model=hidden_dim,
                           d_state=hidden_dim // 2,
                           d_conv=7,
                           )
        self.spatial_attn_1 = nn.Sequential(nn.Conv1d(2*hidden_dim,
                                      hidden_dim // 2,
                                      kernel_size=7,
                                      padding='same'),
                                      nn.ELU())
        
        self.spatial_attn_2 = nn.Sequential(nn.Conv1d(hidden_dim,
                                      hidden_dim // 4,
                                      kernel_size=7,
                                      dilation=3,
                                      padding='same'),
                                      nn.ELU())
        
        self.out = nn.Sequential(nn.Conv1d(hidden_dim // 4, 1,kernel_size=1),
                                 )
    def forward(self, audio):

        o = self.input_l(audio)
        
        o = self.mamba(o.permute(0,-1,1)).permute(0, -1, 1)
        o = o + audio
        max_pool = F.max_pool1d(o, kernel_size=2)
        avg_pool = F.avg_pool1d(o, kernel_size=2)
        
        o = self.spatial_attn_1(torch.cat([max_pool, avg_pool], dim=1))
        
        max_pool = F.max_pool1d(o, kernel_size=2)
        avg_pool = F.avg_pool1d(o, kernel_size=2)
        o = self.spatial_attn_2(torch.cat([max_pool, avg_pool], dim=1))
        

"""mag = torch.rand(4, 1, 1024, 1025).to('cuda')
noisy_phase = torch.rand(4, 1, 1024, 1025).to('cuda')
phs = MambaPhaseCorrector().to('cuda')
phs(mag, noisy_phase).shape"""

audio = torch.rand(1,1,128000).to('cuda')
e = EncoderAudio().to('cuda')
e(audio)

torch.Size([1, 256, 128000])
torch.Size([1, 256, 128000])
torch.Size([1, 128, 64000])
torch.Size([1, 64, 32000])


In [ ]:
class EncoderAudio(nn.Module):
    def __init__(self, kernel_size = 3, hidden_dim = 256, out_dim=512, ):
        super().__init__()
        self.input_l = nn.Conv1d(1, hidden_dim, kernel_size=1)
        self.mamba = Mamba(d_model=hidden_dim,
                           d_state=hidden_dim // 2,
                           d_conv=7,
                           )
        self.spatial_attn_1 = nn.Sequential(nn.Conv1d(2*hidden_dim,
                                      hidden_dim // 2,
                                      kernel_size=7,
                                      padding='same'),
                                      nn.ELU())
        
        self.spatial_attn_2 = nn.Sequential(nn.Conv1d(hidden_dim,
                                      hidden_dim // 4,
                                      kernel_size=7,
                                      dilation=3,
                                      padding='same'),
                                      nn.ELU())
        
        self.out = nn.Sequential(nn.Conv1d(hidden_dim // 4, 1,kernel_size=1),
                                 )
    def forward(self, audio):

        o = self.input_l(audio)
        
        o = self.mamba(o.permute(0,-1,1)).permute(0, -1, 1)
        o = o + audio
        max_pool = F.max_pool1d(o, kernel_size=2)
        avg_pool = F.avg_pool1d(o, kernel_size=2)
        
        o = self.spatial_attn_1(torch.cat([max_pool, avg_pool], dim=1))
        
        max_pool = F.max_pool1d(o, kernel_size=2)
        avg_pool = F.avg_pool1d(o, kernel_size=2)
        o = self.spatial_attn_2(torch.cat([max_pool, avg_pool], dim=1))

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from mamba_ssm import Mamba


class MambaBlock(nn.Module):
    def __init__(self, d_model=256, out_dim=512):
        super().__init__()
        self.input = nn.Conv1d(1, d_model, kernel_size=1)
        self.mamba = Mamba(d_model=d_model, d_state=d_model // 2, d_conv=7)
        self.group_norm = nn.GroupNorm(4, d_model)
        self.spatial_attention = nn.Sequential(
            nn.Conv1d(d_model, out_dim, kernel_size=7, padding='same'),
            nn.Softmax(dim=-1)  
        )
    
    def forward(self, x):
        residual = x 
        x = self.input(x)
        out = self.mamba(x.permute(0, -1, 1)).permute(0, -1, 1)
        out = self.group_norm(out + x)
        out = self.spatial_attention(out) 
        return F.max_pool1d(out + residual, kernel_size=2)  
    
class Conv1dBlock(nn.Module):
    def __init__(self, out_channels=512, kernel_size=3, out_dim=512, num_groups=32):
        super().__init__()
        self.input = nn.Conv1d(1, out_channels, kernel_size=kernel_size, padding='same', dilation=3)
        
        
        self.conv_dilation = nn.Conv1d(
            out_channels, out_channels, 
            kernel_size=kernel_size, padding='same', dilation=3
        )
        self.activation = nn.Sequential(
            nn.ELU(),
            nn.Dropout1d(0.5),
            nn.GroupNorm(4, out_channels)
        )
        
        
        self.depthwise = nn.Conv1d(
            out_channels, out_channels, 
            kernel_size=1, groups=num_groups
        )
        self.pointwise = nn.Conv1d(
            out_channels, out_dim, 
            kernel_size=kernel_size, padding='same'
        )
        

        self.spatial_attention = nn.Sequential(
            nn.Conv1d(out_dim, out_dim, kernel_size=3, padding='same'),
            nn.Softmax(dim=-1) 
        )
    
    def forward(self, x):
        residual = x
        x = self.input(x)
        
        out = self.conv_dilation(x)
        out = self.activation(out + x)  
        
        
        out = self.depthwise(out)
        out = self.pointwise(out)
        out = self.activation(out) + out
        
        attention = self.spatial_attention(out)
        
        return F.max_pool1d(attention + residual, kernel_size=2)  


class AudioEncoder(nn.Module):
    def __init__(self, d_model=8, out_dim=256, kernel_size=3, num_groups=8,
                 out_size = (4, 4)):
        super().__init__()

        self.mamba_block = MambaBlock(d_model,out_dim)
        self.conv_block = Conv1dBlock(out_dim, kernel_size, out_dim, num_groups)
        self.out = nn.Sequential(nn.Conv1d(out_dim * 2, out_dim, kernel_size=1),
                                  nn.MaxPool1d(kernel_size=4))
    
        self.avg_pooler = nn.AdaptiveAvgPool2d(out_size)
    
    def forward(self, x):

        out = self.out(torch.cat([self.mamba_block(x), self.conv_block(x)], dim=1)).unsqueeze(-1)
        out = self.avg_pooler(out)
        return out 
    


        

KeyboardInterrupt: 

In [ ]:
import cpuinfo
from time import perf_counter
from loaders import *
from lightning_modules.lightning_module import *
from utils.cfg_loader import load_cfg
import GPUtil

cfg_path = 'configs/denoise_model_v2_cfg.yaml'
model_cfg = load_cfg(cfg_path)

with torch.no_grad():
    clean_input = torch.randn(1, 1, 128000).to('cuda')
    noise_input = torch.randn(1, 1, 128000).to('cuda')
    m = UltraMaxSpectrogramLightningModelUnet(**model_cfg).to('cuda')
    m._step([clean_input, noise_input], 'train')


/home/ys/diploma/denoising_env/lib/python3.12/site-packages/lightning/pytorch/core/module.py:441: You are trying to `self.log()` but the `self.trainer` reference is not registered on the model yet. This is most likely because the model hasn't been passed to the `Trainer`


In [3]:
#torch.cuda.empty_cache()
!nvidia-smi

Mon Apr 21 18:36:54 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 565.72                 Driver Version: 566.14         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650        On  |   00000000:01:00.0 Off |                  N/A |
| N/A   67C    P0             16W /   50W |     241MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch.nn as nn
import torch
import torchaudio.functional as F
import torchaudio.transforms as T

class SiSDRLoss(nn.Module):
    def __init__(self, eps: float = 1e-9):
        """

        Args:
            eps (float): eps for stabibiluty calculations. Defaults to 1e-9.
        """
        super().__init__()
        self.eps = eps


    def forward(self, output, target):

        alpha = torch.sum(output * target, dim=-1,keepdim=True) / torch.norm(target, dim=-1)**2 

        proj = alpha * target

        proj_norm = torch.norm(proj, dim=-1)
        diff_norm = torch.norm((proj - output), dim=-1)

        return  -(10 * (torch.log10(proj_norm**2 / (diff_norm**2 + self.eps )))).mean()

class MultiResolutionLoss(nn.Module):
    def __init__(self, n_ftts: list = [256, 512, 1025, 2096]):
        super().__init__()
        self.nftts = n_ftts
    def forward(self, clean, enchanced):

        return torch.mean(torch.stack([nn.functional.l1_loss(T.Spectrogram(n_fft=n_fft, 
                                                                           power=1.0,
                                                                           normalized=True)(clean).log1p(),
                                                           T.Spectrogram(n_fft=n_fft, 
                                                                         power=1.0,
                                                                         normalized=True)(enchanced).log1p()) for n_fft in self.nftts]))

mrsl = MultiResolutionLoss()
sisdr = SiSDRLoss()

mrsl(aud1[:, 0 ,None,...], result[:, 0 ,None,...]), sisdr(aud1[:, 0 ,None,...], result[:, 0 ,None,...]) 

RuntimeError: stft input and window must be on the same device but got self on cuda:0 and window on cpu